## Setup

In [20]:
%cd /home/ubuntu/Project/libero_development

/home/ubuntu/Project/libero_development


/home/ubuntu/pyvenv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [21]:
from src.launch_cluster import launch_cluster, shutdown_cluster
cluster, client = launch_cluster(8)

Inizializzazione del cluster SSH con 8 worker...
Worker selezionati: ['10.67.22.254', '10.67.22.34', '10.67.22.145', '10.67.22.121', '10.67.22.192', '10.67.22.18', '10.67.22.187', '10.67.22.48']


2026-08-25 13:21:16,250 - distributed.deploy.ssh - INFO - 2026-08-25 13:21:16,249 - distributed.http.proxy - INFO - To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
2026-08-25 13:21:16,252 - distributed.deploy.ssh - INFO - /home/ubuntu/pyvenv/lib/python3.10/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
2026-08-25 13:21:16,252 - distributed.deploy.ssh - INFO - Perhaps you already have a cluster running?
2026-08-25 13:21:16,253 - distributed.deploy.ssh - INFO - Hosting the HTTP server on port 42139 instead
2026-08-25 13:21:16,253 - distributed.deploy.ssh - INFO - warnings.warn(
2026-08-25 13:21:16,278 - distributed.deploy.ssh - INFO - 2026-08-25 13:21:16,277 - distributed.scheduler - INFO - State start
2026-08-25 13:21:16,281 - distributed.deploy.ssh - INFO - 2026-08-25 13:21:16,281 - distributed.scheduler - INFO - Closing scheduler. Reason: failure-to-start-<class 'OSError'>
2026

RuntimeError: Cluster failed to start: Worker failed to start

#### IF ALREADY EXISTING CLUSTER:

In [22]:
from dask.distributed import Client

SCHEDULER_ADDRESS = "tcp://10.67.22.194:8786"

try:
    client = Client(SCHEDULER_ADDRESS, timeout="10s")
    print("Connected to cluster successfully!")
    print(f"Dask Dashboard link: {client.dashboard_link}")

except Exception as e:
    print(f"Connection error: {e}")

Connected to cluster successfully!
Dask Dashboard link: http://10.67.22.194:8787/status


2026-08-25 13:23:06,515 - distributed.client - ERROR - Failed to reconnect to scheduler after 10.00 seconds, closing client


## Load dataset (10%)

In [23]:
from src.data_loader import load_dataset

DATASET_URL_10PC = "https://ndownloader.figshare.com/files/5976042"
RAW = "/home/ubuntu/Project/libero_development/data/kddcup_data.gz"   # già in cache, niente download
PQ  = "/tmp/kddcup_data.parquet"
COL_NAMES = [
    "duration","protocol_type","service","flag","src_bytes",
    "dst_bytes","land","wrong_fragment","urgent","hot",
    "num_failed_logins","logged_in","num_compromised","root_shell",
    "su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login",
    "is_guest_login","count","srv_count","serror_rate",
    "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
    "diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate",
    "dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate",
    "dst_host_srv_rerror_rate","label"
]

In [24]:
X_bag, (mean_ar, std_ar) = load_dataset(
    DATASET_URL_10PC, RAW, PQ, PQ, COL_NAMES,
    n_partitions=64, client=client, force_download=True
)

Converting .gz -> Parquet chunk sizes...
Parquet file created (compressed with snappy).
Number of partitions before preprocessing: 64
Constant columns: ['num_outbound_cmds', 'is_host_login']
Computing global mean and std (first pass over data)...
Distributed bag created with 32 partitions.
Number of samples: 494021


## Run experiments

#### Varying number of partitions

In [25]:
from src.benchmark import run_benchmark

In [ ]:
#delayed_parts = [dask.delayed(np.vstack)(p) for p in X_bag.to_delayed()]
#X_arr = np.vstack(client.gather(client.compute(delayed_parts)))
#print("X:", X_arr.shape)

R = 10
combos = [                      # identiche alla storica
    (8, 32, 1, R),   # under-partitioned
    (8, 64, 1, R),   # balanced (1 part/thread)
    (8, 65, 1, R),   # imbalanced
    (8, 128, 1, R),  # over-partitioned
]
df = run_benchmark(client, X_arr=X_arr, combinations=combos,
                   k_values=[1000], label="b1_validation",
                   max_iter_fit=10, seed=42, averaging_iterations=10)
df

### Tables 3 and 4

In [19]:
from src.paper_experiments import run_table34, table34_cost_table, table34_time_table

### Figure 5.1

### Figure 5.2

## Cluster shutdown

In [5]:
shutdown_cluster(cluster, client)

Cluster e client chiusi.
